# EDA Profundo - Horno B3001

## Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi':110,'font.size':9,'axes.grid':True,'grid.alpha':0.3})

## Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/Proyecto_B3001')
XLSX=BASE/'Dataset_T_C_B-3001.xlsx'
OUT=BASE/'outputs'; OUT.mkdir(exist_ok=True)

In [ ]:
# 0. PARÁMETROS DE ANÁLISIS

In [ ]:
SAT_LIMIT = 700.0     # >700 °C = saturación de señal (clamp del historiador)
RUN_TH    = 250.0     # referencia >= 250 °C  -> horno en operación
STOP_TH   = 150.0     # referencia <  150 °C  -> horno detenido
MIN_STOP_H   = 4      # horas mínimas en frío para considerar detención
GAP_CLOSE_H  = 6      # huecos <= 6 h dentro de una detención se cierran
MERGE_STOP_H = 24     # detenciones separadas por < 24 h se fusionan

In [ ]:
# 1. MAPEO DE TAGS
#    OJO: en el archivo original los pasos 3 y 4 vienen rotulados como "P1.1"
#    y "P2.1" (etiquetas duplicadas de P1/P2). Aquí se reasignan a P3 y P4.

In [ ]:
MAPPING = {
    # PASO 1
    "TI_30021.pv": ("P1", "CONV"),
    "TI_30017A.pv": ("P1", "T1 SP"),  "TI_30017D.pv": ("P1", "T1 SO"),
    "TI_30017B.pv": ("P1", "T5 SP"),  "TI_30017E.pv": ("P1", "T5 SO"),
    "TI_30017C.pv": ("P1", "T10 NP"), "TI_30017F.pv": ("P1", "T10 NO"),
    # PASO 2
    "TI_30022.pv": ("P2", "CONV"),
    "TI_30018A.pv": ("P2", "T1 NP"),  "TI_30018D.pv": ("P2", "T1 NO"),
    "TI_30018B.pv": ("P2", "T5 NP"),  "TI_30018E.pv": ("P2", "T5 NO"),
    "TI_30018C.pv": ("P2", "T10 SP"), "TI_30018F.pv": ("P2", "T10 SO"),
    # PASO 3  (rotulado "P1.1" en el archivo)
    "TI_30033.pv": ("P3", "CONV"),
    "TI_30031A.pv": ("P3", "T1 SP"),  "TI_30031D.pv": ("P3", "T1 SO"),
    "TI_30031B.pv": ("P3", "T5 SP"),  "TI_30031E.pv": ("P3", "T5 SO"),
    "TI_30031C.pv": ("P3", "T10 NP"), "TI_30031F.pv": ("P3", "T10 NO"),
    # PASO 4  (rotulado "P2.1" en el archivo)
    "TI_30034.pv": ("P4", "CONV"),
    "TI_30032A.pv": ("P4", "T1 NP"),  "TI_30032D.pv": ("P4", "T1 NO"),
    "TI_30032B.pv": ("P4", "T5 NP"),  "TI_30032E.pv": ("P4", "T5 NO"),
    "TI_30032C.pv": ("P4", "T10 SP"), "TI_30032F.pv": ("P4", "T10 SO"),
    # LAZOS DE CONTROL
    "TC_30019.pv": ("CTRL", "P1"), "TC_30020.pv": ("CTRL", "P2"),
    "TC_30027.pv": ("CTRL", "P3"), "TC_30028.pv": ("CTRL", "P4"),
}
TAGS  = list(MAPPING)
PASOS = ["P1", "P2", "P3", "P4"]
CONV  = {p: c for c, (pp, n) in MAPPING.items() if n == "CONV" for p in [pp]}
TUBES = {p: [c for c, (pp, n) in MAPPING.items() if pp == p and n != "CONV"]
         for p in PASOS}
ALL_TUBES = [c for p in PASOS for c in TUBES[p]]
name = lambda c: f"{MAPPING[c][0]} {MAPPING[c][1]}"

In [ ]:
# 2. CARGA Y LIMPIEZA

In [ ]:
def cargar_y_limpiar(path=XLSX):
    """Lee el xlsx, descarta las 3 filas de metadatos, coerciona a numérico,
    deduplica timestamps y reindexa a grilla horaria continua.

    Devuelve: (df_num, sat, auditoria_no_numericos)
      df_num : temperaturas crudas (saturación INCLUIDA)
      sat    : máscara booleana de saturación (>700 °C)
    """
    df = pd.read_excel(path, sheet_name=0, header=0)

    # filas 0-2 = descripción / unidad / paso  →  se descartan
    df = df.iloc[3:].reset_index(drop=True)
    df = df.rename(columns={df.columns[0]: "ts"})
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

    # --- auditoría de etiquetas no numéricas ANTES de coercionar -------------
    auditoria = {}
    for c in TAGS:
        num = pd.to_numeric(df[c], errors="coerce")
        bad = df[c][num.isna() & df[c].notna()]
        if len(bad):
            auditoria[c] = bad.astype(str).value_counts().to_dict()

    # --- toda etiqueta no numérica -> NaN -----------------------------------
    #   (típicamente "[-11059] No Good Data For Calculation" del historiador)
    for c in TAGS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- eje temporal: dedup + grilla horaria continua ----------------------
    df = (df.dropna(subset=["ts"])
            .drop_duplicates(subset="ts", keep="first")
            .sort_values("ts")
            .set_index("ts"))
    grilla = pd.date_range(df.index.min(), df.index.max(), freq="h")
    df = df.reindex(grilla)[TAGS]

    # --- saturación ---------------------------------------------------------
    sat = df > SAT_LIMIT
    return df, sat, auditoria


def reporte_calidad(df, sat, auditoria):
    """NaN %, horas saturadas y etiquetas encontradas, por tag."""
    vals = df.where(~sat)          # temps sin saturación
    q = pd.DataFrame({
        "nan_pct":   (vals.isna() & ~sat).mean() * 100,
        "sat_horas":  sat.sum(),
        "etiquetas":  [sum(auditoria.get(c, {}).values()) for c in df.columns],
        "min":        vals.min(),
        "max":        vals.max(),
    })
    q.index = [name(c) for c in q.index]
    return q.round(1)

In [ ]:
# 3. ESTADO OPERACIONAL Y DETECCIÓN DE DETENCIONES

In [ ]:
def referencia_operacion(vals):
    """Señal robusta de estado del horno: mediana de las 4 CONV,
    con respaldo en la mediana de todos los tubos cuando las CONV son NaN."""
    conv = vals[[CONV[p] for p in PASOS]].median(axis=1)
    return conv.fillna(vals[ALL_TUBES].median(axis=1))


def clasificar_estado(op_ref):
    est = np.where(op_ref >= RUN_TH, "RUN",
          np.where(op_ref <  STOP_TH, "STOP", "TRANS"))
    est = pd.Series(est, index=op_ref.index)
    est[op_ref.isna()] = "NODATA"
    return est


def _episodios(mask, min_h=1):
    """Tramos contiguos True de una serie booleana -> [(ini, fin, horas)]."""
    grp = (mask != mask.shift()).cumsum()
    out = []
    for _, idx in mask.groupby(grp).groups.items():
        if mask.loc[idx].iloc[0] and len(idx) >= min_h:
            out.append((idx[0], idx[-1], len(idx)))
    return out


def detectar_detenciones(estado, op_ref):
    """STOP contiguo, cerrando huecos cortos y fusionando eventos vecinos."""
    frio = (estado == "STOP").astype(int)

    # dilatación: cierra interrupciones <= GAP_CLOSE_H dentro de una detención
    dilatado = frio.rolling(2 * GAP_CLOSE_H + 1, center=True, min_periods=1).max()

    eps = []
    grp = (dilatado.diff() != 0).cumsum()
    for _, idx in dilatado.groupby(grp).groups.items():
        if dilatado.loc[idx].iloc[0] == 1 and frio.loc[idx].sum() >= MIN_STOP_H:
            eps.append([idx[0], idx[-1], op_ref.loc[idx].min()])

    # fusión de episodios separados por < MERGE_STOP_H
    fus = []
    for ini, fin, tmin in sorted(eps):
        if fus and (ini - fus[-1][1]).total_seconds() / 3600 < MERGE_STOP_H:
            fus[-1][1] = fin
            fus[-1][2] = min(fus[-1][2], tmin)
        else:
            fus.append([ini, fin, tmin])

    st = pd.DataFrame(fus, columns=["inicio", "fin", "t_min"])
    st["horas"] = (st.fin - st.inicio).dt.total_seconds() / 3600 + 1
    st["dias"] = st.horas / 24
    return st[["inicio", "fin", "horas", "dias", "t_min"]]


def detectar_trips(estado, stops, min_h=2, ventana="48h"):
    """Tramos TRANS que NO son el enfriamiento/calentamiento de una detención
    → reducciones de carga o trips parciales."""
    tr = pd.DataFrame(_episodios(estado == "TRANS", min_h),
                      columns=["inicio", "fin", "horas"])
    v = pd.Timedelta(ventana)
    cerca = [((stops.inicio - b).abs().min() < v) or ((a - stops.fin).abs().min() < v)
             for a, b in zip(tr.inicio, tr.fin)]
    return tr[~np.array(cerca)].reset_index(drop=True)

In [ ]:
# 4. DESVIACIONES Y EXIGENCIA TÉRMICA

In [ ]:
def estadisticas_tmt(vals, run):
    """P50 / media / std / P95 / max por tubo, solo en operación."""
    s = vals[ALL_TUBES][run].agg(
        ["median", "mean", "std", lambda x: x.quantile(.95), "max"]).T
    s.columns = ["P50", "media", "std", "P95", "max"]
    s.index = [name(c) for c in s.index]
    return s.round(1)


def bandas_tmt(vals, sat, run):
    """Horas de operación por banda de temperatura (exposición acumulada)."""
    b = pd.DataFrame({
        "h_600_650": ((vals[ALL_TUBES] > 600) & (vals[ALL_TUBES] <= 650))[run].sum(),
        "h_650_700": (vals[ALL_TUBES] > 650)[run].sum(),
        "h_saturado": sat[ALL_TUBES][run].sum(),
    })
    b.index = [name(c) for c in b.index]
    return b


def residuales(vals, run, umbral=75):
    """Desvío de cada tubo respecto de la mediana de SU PROPIO paso.
    Aísla el desbalance tubo-a-tubo del nivel general de fuego."""
    res = {}
    for p, cols in TUBES.items():
        med = vals[cols].median(axis=1)
        for c in cols:
            res[c] = vals[c] - med
    res = pd.DataFrame(res)[run]

    rs = res.agg(["mean", "std", lambda s: (s.abs() > umbral).mean() * 100]).T
    rs.columns = ["sesgo_medio", "std", f"pct_abs_gt{umbral}"]
    rs.index = [name(c) for c in rs.index]
    return res, rs.round(1)


def episodios_saturacion(sat, min_h=3):
    filas = []
    for c in sat.columns:
        if sat[c].any():
            for a, b, h in _episodios(sat[c], min_h):
                filas.append((name(c), a, b, h))
    return (pd.DataFrame(filas, columns=["sensor", "inicio", "fin", "horas"])
              .sort_values("horas", ascending=False)
              .reset_index(drop=True))

In [ ]:
# 5. FIGURAS

In [ ]:
def fig1_timeline(op_ref, stops):
    fig, ax = plt.subplots(figsize=(13, 3.6))
    d = op_ref.resample("D").mean()
    ax.plot(d.index, d.values, lw=.5, color="#1f4e79")
    for r in stops.itertuples():
        ax.axvspan(r.inicio, r.fin, color="crimson", alpha=.35)
    ax.set_ylabel("T° mediana CONV pasos (°C)")
    ax.set_title("B-3001 · Operación 2008–2026 (rojo = detenciones detectadas)")
    fig.tight_layout(); fig.savefig(OUT / "fig1_timeline.png"); plt.close(fig)


def fig2_disponibilidad(estado, stops):
    avail = estado.groupby(estado.index.year).apply(lambda x: (x == "RUN").mean() * 100)
    horas = stops.groupby(stops.inicio.dt.year).horas.sum().reindex(avail.index, fill_value=0)

    fig, ax = plt.subplots(figsize=(11, 3.6)); ax2 = ax.twinx()
    ax.bar(avail.index, avail.values, color="#2e75b6", alpha=.8, label="% horas en operación")
    ax2.plot(horas.index, horas.values, "o-", color="crimson", label="Horas de detención")
    ax.set_ylim(0, 105); ax.set_ylabel("% RUN"); ax2.set_ylabel("Horas detención/año")
    ax.set_title("Disponibilidad anual y horas de detención")
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="lower left")
    fig.tight_layout(); fig.savefig(OUT / "fig2_disponibilidad.png"); plt.close(fig)


def fig3_saturacion(sat):
    m = sat[ALL_TUBES].groupby(sat.index.year).sum()
    m.columns = [name(c) for c in m.columns]
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(m.T.values, aspect="auto", cmap="Reds")
    ax.set_xticks(range(len(m.index)), m.index, rotation=45)
    ax.set_yticks(range(len(m.columns)), m.columns, fontsize=7)
    fig.colorbar(im, label="Horas saturadas (>700 °C) por año")
    ax.set_title("Saturación de señal por sensor y año"); ax.grid(False)
    fig.tight_layout(); fig.savefig(OUT / "fig3_saturacion.png"); plt.close(fig)


def fig4_ranking(stats):
    s = stats.sort_values("P95")
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(s.index, s.P95, color=["crimson" if v > 640 else "#2e75b6" for v in s.P95])
    ax.axvline(650, ls="--", color="k", lw=1); ax.text(651, .2, "650 °C", fontsize=8)
    ax.set_xlim(450, 700); ax.set_xlabel("P95 TMT en operación (°C)")
    ax.set_title("Tubos más exigidos térmicamente (P95, 2008–2026)")
    fig.tight_layout(); fig.savefig(OUT / "fig4_ranking.png"); plt.close(fig)


def fig5_paradas(op_ref, ventanas=(("2021-10-01", "2022-01-20", "Parada mayor: oct-2021 → ene-2022 (~91 días)"),
                                   ("2025-03-25", "2025-06-01", "Parada abr–may 2025 (~50 días)"))):
    fig, axs = plt.subplots(1, len(ventanas), figsize=(12, 3.6))
    for ax, (a, b, t) in zip(np.atleast_1d(axs), ventanas):
        w = op_ref[a:b]
        ax.plot(w.index, w.values, lw=.7, color="#1f4e79")
        ax.set_title(t); ax.set_ylabel("°C")
    fig.tight_layout(); fig.savefig(OUT / "fig5_paradas.png"); plt.close(fig)


def fig6_tendencia(vals, run):
    """P95 anual del MÁXIMO del paso: deriva de largo plazo del nivel térmico."""
    tr = pd.DataFrame({p: (lambda m: m.groupby(m.index.year).quantile(.95))(
                          vals[cols][run].max(axis=1))
                       for p, cols in TUBES.items()})
    fig, ax = plt.subplots(figsize=(10, 3.8))
    for p in tr:
        ax.plot(tr.index, tr[p], "o-", label=p)
    ax.set_ylabel("P95 del máx TMT del paso (°C)"); ax.legend()
    ax.set_title("Tendencia anual de temperatura máxima de metal por paso")
    fig.tight_layout(); fig.savefig(OUT / "fig6_tendencia.png"); plt.close(fig)
    return tr.round(1)


def fig7_desviaciones(res, sensores=("TI_30032F.pv", "TI_30017F.pv", "TI_30018D.pv")):
    fig, ax = plt.subplots(figsize=(12, 3.6))
    for c in sensores:
        r = res[c].resample("W").median()
        ax.plot(r.index, r.values, lw=.8, label=name(c))
    ax.axhline(0, color="k", lw=.8)
    ax.set_ylabel("Desvío vs mediana del paso (°C)"); ax.legend()
    ax.set_title("Desviaciones sistemáticas de tubos vs su paso (mediana semanal, en operación)")
    fig.tight_layout(); fig.savefig(OUT / "fig7_desviaciones.png"); plt.close(fig)

In [ ]:
# 6. PIPELINE

In [ ]:
def main():
    # --- limpieza -----------------------------------------------------------
    df, sat, auditoria = cargar_y_limpiar()
    vals = df.where(~sat)            # temperaturas SIN saturación (para estadística)
    print(f"Periodo: {df.index.min()} → {df.index.max()}  ({len(df):,} horas)")
    print(f"Celdas saturadas (>{SAT_LIMIT:.0f} °C): {int(sat.values.sum()):,}")
    print("\nEtiquetas no numéricas encontradas:")
    for c, d in list(auditoria.items())[:3]:
        print("  ", name(c), d)

    calidad = reporte_calidad(df, sat, auditoria)
    print("\nPeores por NaN:\n", calidad.sort_values("nan_pct", ascending=False).head(8))

    # --- estado y detenciones ----------------------------------------------
    op_ref = referencia_operacion(vals)
    estado = clasificar_estado(op_ref)
    run = estado == "RUN"

    stops = detectar_detenciones(estado, op_ref)
    trips = detectar_trips(estado, stops)
    print(f"\nDetenciones: {len(stops)} | {stops.horas.sum():,.0f} h "
          f"({stops.horas.sum()/24:,.0f} días) | disponibilidad {run.mean()*100:.1f} %")
    print(f"Trips / reducciones de carga: {len(trips)}")
    print("\nParadas mayores (>30 días):\n",
          stops[stops.dias > 30][["inicio", "fin", "dias"]].to_string(index=False))

    # --- exigencia térmica y desviaciones ----------------------------------
    stats = estadisticas_tmt(vals, run)
    bandas = bandas_tmt(vals, sat, run)
    res, rs = residuales(vals, run)
    sat_ep = episodios_saturacion(sat)
    print("\nTop 6 por P95:\n", stats.sort_values("P95", ascending=False).head(6))
    print("\nMayores desviaciones vs paso:\n",
          rs.sort_values("pct_abs_gt75", ascending=False).head(6))

    # --- figuras ------------------------------------------------------------
    fig1_timeline(op_ref, stops)
    fig2_disponibilidad(estado, stops)
    fig3_saturacion(sat)
    fig4_ranking(stats)
    fig5_paradas(op_ref)
    tendencia = fig6_tendencia(vals, run)
    fig7_desviaciones(res)

    # --- exportes -----------------------------------------------------------
    stops.to_csv(OUT / "detenciones_B3001.csv", index=False)
    trips.to_csv(OUT / "trips_reducciones.csv", index=False)
    sat_ep.to_csv(OUT / "episodios_saturacion.csv", index=False)
    stats.join(bandas).to_csv(OUT / "estadisticas_tubos.csv")
    rs.to_csv(OUT / "desviaciones_tubos.csv")
    calidad.to_csv(OUT / "calidad_datos.csv")
    tendencia.to_csv(OUT / "tendencia_anual_P95.csv")
    print(f"\nListo → {OUT.resolve()}")

    return dict(df=df, vals=vals, sat=sat, op_ref=op_ref, estado=estado,
                stops=stops, trips=trips, stats=stats, res=res)


if __name__ == "__main__":
    main()